#1. Ensemble Learning - Bagging

Let us try to use KNN as the base estimator.
<br>
What can you observe from the results? <br>
Sample solution: KNN may not be the best base estimator for bagging - KNN is a stable algorithm (small changes in data don't drastically change decision boundaries), so bagging may not help much. OR the hyperparameter tuning is needed where the default values are not optimal

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import os

We will again work on titanic dataset.

In [25]:
#step 1: Read in Dataset and save in 'Train'
train = pd.read_csv('C:/Users/YEO/OneDrive/Desktop/Degree/Year 2/SEM 3 LONG SEM (2025)/TML6223-MACHINE LEARNING (ML1,1E)/LAB/Lab7/Lab7/titanic_test.csv')
train.head()
train.info()
train.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Survived     418 non-null    int64  
 2   Pclass       418 non-null    int64  
 3   Name         418 non-null    object 
 4   Sex          418 non-null    object 
 5   Age          332 non-null    float64
 6   SibSp        418 non-null    int64  
 7   Parch        418 non-null    int64  
 8   Ticket       418 non-null    object 
 9   Fare         417 non-null    float64
 10  Cabin        91 non-null     object 
 11  Embarked     418 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 39.3+ KB


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,418.000000,418.000000,418.000000,332.000000,418.000000,418.000000,417.000000
mean,1100.500000,0.363636,2.265550,30.272590,0.447368,0.392344,35.627188
std,120.810458,0.481622,0.841838,14.181209,0.896760,0.981429,55.907576
min,892.000000,0.000000,1.000000,0.170000,0.000000,0.000000,0.000000
25%,996.250000,0.000000,1.000000,21.000000,0.000000,0.000000,7.895800
50%,1100.500000,0.000000,3.000000,27.000000,0.000000,0.000000,14.454200
75%,1204.750000,1.000000,3.000000,39.000000,1.000000,0.000000,31.500000
max,1309.000000,1.000000,3.000000,76.000000,8.000000,9.000000,512.329200


In [32]:
print(train.isnull().sum())
train.info()
print(train.describe())

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Survived     418 non-null    int64  
 2   Pclass       418 non-null    int64  
 3   Name         418 non-null    object 
 4   Sex          418 non-null    object 
 5   Age          332 non-null    float64
 6   SibSp        418 non-null    int64  
 7   Parch        418 non-null    int64  
 8   Ticket       418 non-null    object 
 9   Fare         417 non-null    float64
 10  Cabin        91 non-null     object 
 11  Embarked     418 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 39.3+ KB
   

In [36]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

# Load the dataset into df
df = pd.read_csv('C:/Users/YEO/OneDrive/Desktop/Degree/Year 2/SEM 3 LONG SEM (2025)/TML6223-MACHINE LEARNING (ML1,1E)/LAB/Lab7/Lab7/titanic_train.csv')

# Option 1: If you want to use specific numerical features
numerical_features = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
X = df[numerical_features]
y = df['Survived']

# Option 2: If you want to drop 'PassengerId' and 'Survived' and keep the rest:
# X = df.drop(['PassengerId', 'Survived'], axis=1)
# y = df['Survived']

# Impute missing values in Age using the mean
imputer = SimpleImputer(strategy='mean')
X['Age'] = imputer.fit_transform(X[['Age']])

# Split the data into training and validation sets (70% train, 30% validation)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=42)

C:\Users\YEO\AppData\Local\Temp\ipykernel_13008\194140173.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Age'] = imputer.fit_transform(X[['Age']])


#Option 1: Using KNN as the Base

In [37]:


# Create the full pipeline
model1 = Pipeline(steps=[
    ('preprocessor', ColumnTransformer([
        ('sex_encoder', OrdinalEncoder(), ['Sex'])  # Use OrdinalEncoder for features
    ], remainder='passthrough')),
    ('imputer', SimpleImputer(strategy='mean')),
    ('classifier', BaggingClassifier(
        KNeighborsClassifier(n_neighbors=3),
        n_estimators=20,
        max_samples=0.5,
        random_state=42
    ))
])

#Option 2: Apply Feature Scaling and Tune KNN

In [38]:
from sklearn.preprocessing import StandardScaler

model2 = Pipeline(steps=[
    ('preprocessor', ColumnTransformer([
        ('sex_encoder', OrdinalEncoder(), ['Sex'])
    ], remainder='passthrough')),
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),  # Important for KNN!
    ('classifier', BaggingClassifier(
        KNeighborsClassifier(n_neighbors=5),  # Try different k
        n_estimators=30,
        max_samples=0.7,
        random_state=42
    ))
])

Complete Option 3 and 4
#Option 3: Change the base classifier to decision tree

Modify the Bagging Classifier to DecisionTreeClassifier with the depth is 3. Number of estimator is 50, max sample size is 0.8 with random state is 42.

In [41]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.pipeline import Pipeline

# Create a decision tree classifier as the base estimator
dtree = DecisionTreeClassifier(random_state=42)

# Create the bagging classifier using the decision tree as the base estimator
bagging = BaggingClassifier(estimator=dtree, n_estimators=100, random_state=42)

# Build the pipeline. Here, we only have the bagging classifier, but you could add additional steps if needed.
pipeline = Pipeline([
    ('bagging', bagging)
])

# Assuming X_train and y_train were prepared previously:
pipeline.fit(X_train, y_train)

# For evaluation on the validation set:
y_pred = pipeline.predict(X_val)

# Option 4: Using random forest

In [42]:
from sklearn.ensemble import RandomForestClassifier

# Often works better out-of-the-box
#Step 4: Change Bagging Classifier to Random Forest with number of estimator = 100,
#maximum depth is 5 and random state = 42
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

# Create the Random Forest classifier with 100 trees, max_depth of 5, and random_state 42
rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)

# Build the pipeline. In this case, the pipeline consists of only the Random Forest classifier.
pipeline = Pipeline([
    ('rf', rf)
])

# Fit the pipeline on the training data
pipeline.fit(X_train, y_train)

# Predict on the validation set
y_pred = pipeline.predict(X_val)

In [ ]:
#Step 5: Split data into training and testing sets (30%)


In [ ]:
#Step 6: Train the model


#Step 7: Make predictions


In [ ]:
#Step 8: Evaluate the model

#Alternative method
#you may consider to use a for loop to iterate through the model.item and print each result
# Train and evaluate all models
models = {
    'Bagging with KNN': model1,
    'Feature Scaling and Fine Tune KNN': model2
    #option 3
    #option 4
}

#Use a for loop to call each model (including train, make prediction and evaluation)

Accuracy: 0.8059701492537313

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.89      0.84       157
           1       0.81      0.69      0.75       111

    accuracy                           0.81       268
   macro avg       0.81      0.79      0.79       268
weighted avg       0.81      0.81      0.80       268


Confusion Matrix:
[[139  18]
 [ 34  77]]


In [43]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Assume X and y were prepared previously (features and target)
# For example, if X and y are defined like this:
# X = df.drop(['PassengerId', 'Survived'], axis=1)
# y = df['Survived']
# (And any necessary imputations or transformations have been done)

# Step 5: Split the data into training and testing sets (30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# For demonstration purposes, we assume that model1 and model2 (and any additional options) have been defined.
# Example definitions (replace these with your actual models):
# Let's say model1 is a pipeline that applies bagging with KNN:
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import BaggingClassifier

knn = KNeighborsClassifier()  # default parameters; tune as needed
model1 = Pipeline([
    ('bagging', BaggingClassifier(estimator=knn, n_estimators=10, random_state=42))
])

# And model2 could be a pipeline that applies feature scaling and a fine-tuned KNN:
from sklearn.preprocessing import StandardScaler
model2 = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=5))  # fine-tune n_neighbors or other parameters
])

# You can expand the dictionary with additional models if needed.
models = {
    'Bagging with KNN': model1,
    'Feature Scaling and Fine Tune KNN': model2
    # Option 3: model3,
    # Option 4: model4,
}

# Step 6, 7, & 8: Train the model, make predictions, and evaluate for each model by looping through the dictionary
for name, model in models.items():
    print(f"Evaluating model: {name}")
    
    # Train the model on the training data
    model.fit(X_train, y_train)
    
    # Make predictions on the test set
    y_pred = model.predict(X_test)
    
    # Evaluate the model
    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred)
    confusion = confusion_matrix(y_test, y_pred)
    
    print("Accuracy:", accuracy)
    print("Classification Report:")
    print(report)
    print("Confusion Matrix:")
    print(confusion)
    print("-" * 40)

Evaluating model: Bagging with KNN
Accuracy: 0.667910447761194
Classification Report:
              precision    recall  f1-score   support

           0       0.69      0.80      0.74       157
           1       0.63      0.49      0.55       111

    accuracy                           0.67       268
   macro avg       0.66      0.64      0.64       268
weighted avg       0.66      0.67      0.66       268

Confusion Matrix:
[[125  32]
 [ 57  54]]
----------------------------------------
Evaluating model: Feature Scaling and Fine Tune KNN
Accuracy: 0.6865671641791045
Classification Report:
              precision    recall  f1-score   support

           0       0.71      0.80      0.75       157
           1       0.65      0.53      0.58       111

    accuracy                           0.69       268
   macro avg       0.68      0.66      0.67       268
weighted avg       0.68      0.69      0.68       268

Confusion Matrix:
[[125  32]
 [ 52  59]]
---------------------------------

#Observation from the results
Option 1: Using base KNN, without hyperparameter tuning, no feature scaling, accuracy is approximating 71%

Option 2: Using base KNN, with hyperparameter tuning, feature scaling, accuracy is approximating 77%

Option 3: Using base decisin tree, accuracy is approximating 81%

Option 4: Use random forest, accuracy is approximating 82%

# Ensemble Learning - Boosting

In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

datapath = os.path.join("/content/sample_data/", "")

#read in the data
train = pd.read_csv(datapath + "titanic_train.csv")



ModuleNotFoundError: No module named 'xgboost'

#Features Set

Include numerical featuers and gender.

In [ ]:
#Step 1: Handling the features
#include the Sex and numerical features (no need to perform label encoder)
#assign those features and save in X except 'PassengerID' and 'Survived'
#assign 'Survived' to Y

In [ ]:
preprocessor = ColumnTransformer([ ('sex_encoder', OrdinalEncoder(), ['Sex']),], remainder='passthrough')

#Ada Boost
* Sequentially trains weak learners (typically shallow trees)
* Gives more weight to misclassified samples in each iteration
* Combines all weak learners via weighted majority vote

✅ Fast and simple <br>
✅ Less prone to overfitting than single models <br>
❌ Sensitive to noisy data/outliers

In [6]:
# Option 1: AdaBoost (with Decision Tree as base estimator)
ada_model = Pipeline([
    ('preprocessor', preprocessor),
    ('imputer', SimpleImputer(strategy='median')),
    ('classifier', AdaBoostClassifier(
        n_estimators=100,
        learning_rate=0.5,
        random_state=42
    ))
])

NameError: name 'preprocessor' is not defined

# Gradient Boosting
* Builds trees sequentially to correct previous errors
* Minimizes loss function (e.g., log loss) using gradient descent
* Each tree predicts the residual errors of the combined ensemble

✅ Generally more accurate than AdaBoost
✅ Handles mixed data types well
❌ Requires careful tuning (learning rate, tree depth)

In [ ]:
# Option 2: Gradient Boosting (GBM)
#Step 2: name the model as 'gbm_model', change classifier to Gradient Boosting Classifier
#estimater = 150, learning rate is 0.1, depth is 3 and random state is 42


# XGBoost
* Optimized GBM with regularization (L1/L2)
* Uses advanced techniques like:
     1. Tree pruning
     2. Hardware optimization
     3. Weighted quantile sketching for splits

✅ State-of-the-art accuracy for tabular data <br>
✅ Built-in cross-validation + early stopping <br>
❌ More hyperparameters to tune

In [ ]:
# Option 3: XGBoost (Best for most cases)

#Step 3: name the model as 'gbm_model', change classifier to XGBClassifier
#More parameters to set: estimater = 200, learning rate is 0.05, depth is 3, subsample is 0.9, column sample by tree = 0.8
# random state is 42 and the evaluation metric is logloss


In [ ]:
#Step 4: Split data into training and testing sets (30%)

In [ ]:
# Step 5: Train and evaluate all models
